In [ ]:
import math

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from rotation.symbolic import ALL_SYMBOL, create_3d_rotation, create_mix_equation
from rotation.solver import initialize_system, create_func, solve_mixing, pseudo_calculate_meson_couplings, pseudo_calculate_photon_coupling, pseudo_calculate_bare_kaon, pseudo_calculate_kaon_couplings

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S011
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    ALL_SYMBOL["prop"]["pi0"]: 0.1349768,
    ALL_SYMBOL["prop"]["K0"]: 0.497611,
    ALL_SYMBOL["prop"]["eta"]: 0.547862,
}

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [ALL_SYMBOL["prop"]["pi0"], ALL_SYMBOL["bare"]["pi0"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / PDG[ALL_SYMBOL["prop"]["pi0"]]
    elif key in [ALL_SYMBOL["prop"]["eta"], ALL_SYMBOL["bare"]["eta"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / PDG[ALL_SYMBOL["prop"]["eta"]]
    elif key in [ALL_SYMBOL["prop"]["phi"], ALL_SYMBOL["prop"]["eta0"], ALL_SYMBOL["bare"]["phi"], ALL_SYMBOL["bare"]["pi0-phi"], ALL_SYMBOL["bare"]["eta-phi"], ALL_SYMBOL["bare"]["eta0"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ALL_SYMBOL["angle"].values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

## Prepare solver

In [ ]:
rotate = create_3d_rotation()[0]
equation = create_mix_equation(rotate)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

## Solver inputs

In [ ]:
# Fixed known parameters 
known = {
    ALL_SYMBOL["prop"]["pi0"]: PDG[ALL_SYMBOL["prop"]["pi0"]],
    ALL_SYMBOL["prop"]["eta"]: PDG[ALL_SYMBOL["prop"]["eta"]],
    ALL_SYMBOL["bare"]["phi"]: 1.0,
    ALL_SYMBOL["bare"]["pi0-phi"]: 0.1,
    ALL_SYMBOL["bare"]["eta-phi"]: 0.3,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    ALL_SYMBOL["prop"]["phi"]: 1.0,
    ALL_SYMBOL["bare"]["pi0"]: PDG[ALL_SYMBOL["prop"]["pi0"]],
    ALL_SYMBOL["bare"]["eta"]: PDG[ALL_SYMBOL["prop"]["eta"]],
    ALL_SYMBOL["angle"]["alpha"]: 0.0,
    ALL_SYMBOL["angle"]["beta"]: 0.0,
    ALL_SYMBOL["angle"]["gamma"]: 0.0,
}

# Printout the system with our known values, and unknowns blank
print("Arguments input:")
for k in argument_order:
    if k in known:
        sym.Eq(k, known[k], evaluate=False)
    else:
        sym.Eq(k, sym.Symbol("?"), evaluate=False)
equation.subs(known)

## Run solver

In [ ]:
# Create function
func = create_func(
    system_of_equations,
    argument_order,
    known,
)
try:
    # Solve system
    solved = solve_mixing(
        func,
        argument_order,
        unknown,
    )
    print("Solver succeeded!")
except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")

## Solution display

In [ ]:
# Printout the system with our solved unknowns, and knowns blank
print("Arguments solved:")
for k, v in solved.items():
    v = format_output(k, v)
    sym.Eq(k, v, evaluate=False)

print("The solved equation:")
equation.subs(solved)

print("Compare LHS and RHS evaluations:")
sym.Eq(
    (1*equation.lhs.subs(known|solved)).evalf(),
    (1*equation.rhs.subs(known|solved)).evalf(),
    evaluate=False
)

## Calculate Couplings

In [ ]:
meson_couplings = pseudo_calculate_meson_couplings(
    bare_mediator_dm_coupling = 0.2,
    alpha = solved[ALL_SYMBOL["angle"]["alpha"]],
    beta = solved[ALL_SYMBOL["angle"]["beta"]],
    gamma = solved[ALL_SYMBOL["angle"]["gamma"]],
)
for s,g in zip([ALL_SYMBOL["rotated"][k] for k in ["pi-chi", "phi-chi", "eta-chi"]], meson_couplings):
    sym.Eq(s,g,evaluate=False)

In [ ]:
photon_coupling = pseudo_calculate_photon_coupling(
    known_pion_photon_coupling = 0.15,
    known_eta_photon_coupling = 0.05,
    alpha = solved[ALL_SYMBOL["angle"]["alpha"]],
    beta = solved[ALL_SYMBOL["angle"]["beta"]],
    gamma = solved[ALL_SYMBOL["angle"]["gamma"]],
)
sym.Eq(ALL_SYMBOL["rotated"]["phi-gamma"], photon_coupling, evaluate=False)

In [ ]:
bare_kaon_couplings = pseudo_calculate_bare_kaon(
    energy_pion = 1,
    mass_pion = PDG[ALL_SYMBOL["prop"]["pi0"]],
    mass_kaon = PDG[ALL_SYMBOL["prop"]["K0"]],
)
for s,g in zip([ALL_SYMBOL["coupling"][k] for k in ["pi-pi-K", "phi-pi-K", "eta-pi-K"]], bare_kaon_couplings):
    sym.Eq(s,g,evaluate=False)

print("After rotating:")
kaon_couplings = pseudo_calculate_kaon_couplings(
    bare_pion_pion_kaon_coupling = bare_kaon_couplings[0],
    bare_mediator_pion_kaon_coupling = bare_kaon_couplings[1],
    bare_eta_pion_kaon_coupling = bare_kaon_couplings[2],
    alpha = solved[ALL_SYMBOL["angle"]["alpha"]],
    beta = solved[ALL_SYMBOL["angle"]["beta"]],
    gamma = solved[ALL_SYMBOL["angle"]["gamma"]],
)
for s,g in zip([ALL_SYMBOL["rotated"][k] for k in ["pi-pi-K", "phi-pi-K", "eta-pi-K"]], kaon_couplings):
    sym.Eq(s,g,evaluate=False)